# Insurance Claims as a Workflow

Companion notebook for **lesson P2** of the study notes. The same claims, data and tools as the agent in lesson P1, but here **your code fixes the steps**. The LLM is used once, for the one thing code can't do: read the claim text.

1. **Extract** (LLM): incident type and damaged areas, checked by a gate, retried with feedback
2. **Check coverage** (code): not covered → `needs_review`
3. **Price the damage** (code): the price table
4. **Severity** (code): from the priced total
5. **Route** (code): the routing rules

Lesson P3 compares this with the prompt chain (4.4) and the agent (P1).

**Run it in Colab, setup cell first.** It uses Colab's built-in models (no API key). The setup cell lists the available models; to choose one, uncomment its `MODEL = ...` line there.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · Data and tools (the same as lesson P1)

Policy table, price table and routing rules, unchanged. In P1 the agent chose when to call them; here the workflow calls them in a fixed order.

In [1]:
import re
from typing import List, Literal
from pydantic import BaseModel, Field, ValidationError

POLICIES = {   # stand-in for the policy system
    "C001": {"active": True,  "coverages": ["glass", "collision"],         "deductible": 0},
    "C002": {"active": True,  "coverages": ["collision"],                  "deductible": 500},
    "C003": {"active": True,  "coverages": ["collision", "comprehensive"], "deductible": 1000},
    "C004": {"active": True,  "coverages": ["comprehensive"],              "deductible": 500},
    "C005": {"active": True,  "coverages": ["collision"],                  "deductible": 500},
    "C006": {"active": False, "coverages": ["collision", "comprehensive"], "deductible": 500},
}
COVERAGE_FOR = {"glass": ["glass", "comprehensive"], "collision": ["collision"],
                "hail": ["comprehensive"], "vandalism": ["comprehensive"], "theft": ["comprehensive"]}

def check_coverage(claim_id: str, incident_type: str) -> dict:
    """Is the policy active, and does it cover this kind of incident?"""
    if claim_id not in POLICIES:
        raise ValueError(f"no policy found for claim {claim_id}")
    policy = POLICIES[claim_id]
    covered = policy["active"] and any(c in policy["coverages"] for c in COVERAGE_FOR[incident_type])
    return {"policy_active": policy["active"], "covered": covered, "deductible": policy["deductible"]}

REPAIR_COST = {"windshield": 300, "glass": 250, "front": 3000, "rear": 1200, "side": 800,
               "roof": 1800, "hood": 1500, "door": 1100, "bumper": 900, "fender": 700,
               "quarter panel": 1400, "trunk": 1300, "mirror": 300, "light": 350, "engine": 6000}

def price_damage(areas: list) -> dict:
    """Typical repair cost per damaged area. An area damaged twice (two doors) is listed twice."""
    return {"items": [[a, REPAIR_COST[a]] for a in areas], "total": sum(REPAIR_COST[a] for a in areas)}

COST_RANGES = {"Minor": (100, 1_000), "Moderate": (1_000, 5_000), "Major": (5_000, 50_000)}
GLASS_ONLY = {"windshield", "glass"}

def route_claim(severity: str, est_cost: float, areas: list) -> dict:
    """The routing rules from lesson 4.4. Refuses a cost that doesn't fit the severity."""
    low, high = COST_RANGES[severity]
    if not low <= est_cost <= high:
        raise ValueError(f"{severity} damage must cost {low:,}-{high:,} USD, got {est_cost:,.0f}")
    if severity == "Major":
        return {"queue": "total_loss"}
    if severity == "Moderate":
        return {"queue": "material_damage"}
    return {"queue": "glass" if set(areas) <= GLASS_ONLY else "fast_track"}

## 2 · The one LLM step: extraction

The model reads the claim and returns the facts the code needs. Pydantic is the gate: an incident type or area outside the allowed lists is refused, and the error goes back to the model for another try (a feedback loop, capped at 3 attempts).

In [2]:
Area = Literal["windshield", "glass", "front", "rear", "side", "roof", "hood", "door", "bumper",
               "fender", "quarter panel", "trunk", "mirror", "light", "engine"]

class ClaimFacts(BaseModel):                      # what the extraction must return
    claim_id: str = Field(..., min_length=2, max_length=10)
    incident_type: Literal["glass", "collision", "hail", "vandalism", "theft"]
    damage_areas: List[Area] = Field(..., min_length=1)

EXTRACT_PROMPT = """
You read First Notice of Loss (FNOL) reports for an auto insurer. Return only a JSON object:
{"claim_id": "...", "incident_type": "...", "damage_areas": [...]}
incident_type: glass, collision, hail, vandalism or theft. A car hit while parked is a collision.
damage_areas uses only: windshield, glass, front, rear, side, roof, hood, door, bumper, fender,
quarter panel, trunk, mirror, light, engine. List an area once per damaged part,
e.g. both doors: ["door", "door"].
""".strip()

class GateFailed(Exception):
    pass

def strip_fences(text: str) -> str:
    """Models often wrap JSON in a Markdown fence; remove it before parsing."""
    return re.sub(r"^\s*`{3}(?:json)?\s*\n|\n?`{3}\s*$", "", text.strip())

def extract(fnol: str, max_attempts: int = 3):
    """Returns (facts, llm_calls). Each failed attempt sends the gate's error back to the model."""
    messages = [{"role": "system", "content": EXTRACT_PROMPT}, {"role": "user", "content": fnol}]
    for attempt in range(1, max_attempts + 1):
        raw = strip_fences(get_completion(messages))
        try:
            return ClaimFacts.model_validate_json(raw), attempt
        except ValidationError as e:
            if attempt == max_attempts:
                raise GateFailed(f"extraction failed {max_attempts}x: {e.errors()[0]['msg']}")
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"Your answer failed validation:\n{e}\nReturn corrected JSON only."}]

## 3 · The workflow

Every step after extraction is code, in a fixed order, with one branch (not covered). Severity comes from the priced total instead of a model's judgment. `steps` records what ran, to compare with the agent's path in P1.

In [3]:
def severity_for(total: float) -> str:
    return "Minor" if total <= 1_000 else "Moderate" if total <= 5_000 else "Major"

def decide(facts: ClaimFacts) -> dict:
    """Everything after extraction: plain code, the same path every time."""
    steps = ["check_coverage"]
    coverage = check_coverage(facts.claim_id, facts.incident_type)
    if not coverage["covered"]:
        return {"queue": "needs_review", "reason": "not covered" if coverage["policy_active"] else "policy not active",
                "steps": steps}
    total = price_damage(facts.damage_areas)["total"]
    severity = severity_for(total)
    steps += ["price_damage", "route_claim"]
    try:
        queue = route_claim(severity, total, facts.damage_areas)["queue"]
    except ValueError as e:                              # e.g. above the Major range
        return {"queue": "needs_review", "reason": str(e), "steps": steps}
    return {"queue": queue, "severity": severity, "est_cost": total, "steps": steps}

def triage(fnol: str) -> dict:
    try:
        facts, llm_calls = extract(fnol)
    except GateFailed as e:
        return {"queue": "needs_review", "reason": str(e), "steps": ["extract"], "llm_calls": 3}
    return {"claim_id": facts.claim_id, **decide(facts), "llm_calls": llm_calls}

## 4 · You play the extraction step

Everything after extraction runs without a model. Give `decide` the facts a correct extraction would return, and the workflow produces the same queues the agent should reach in P1:

In [4]:
correct_facts = [
    ClaimFacts(claim_id="C001", incident_type="glass",     damage_areas=["windshield"]),
    ClaimFacts(claim_id="C002", incident_type="collision", damage_areas=["bumper", "light"]),
    ClaimFacts(claim_id="C003", incident_type="collision", damage_areas=["front", "hood", "bumper", "engine"]),
    ClaimFacts(claim_id="C004", incident_type="hail",      damage_areas=["hood", "roof", "trunk", "mirror", "mirror", "glass"]),
    ClaimFacts(claim_id="C005", incident_type="vandalism", damage_areas=["door", "door"]),
    ClaimFacts(claim_id="C006", incident_type="collision", damage_areas=["bumper", "trunk"]),
]
for facts in correct_facts:
    r = decide(facts)
    detail = f"{r['severity']:8} ${r['est_cost']:>7,.0f}" if "severity" in r else r["reason"]
    print(f"{facts.claim_id}  {r['queue']:16} {detail:22} {' > '.join(r['steps'])}")

C001  glass            Minor    $    300      check_coverage > price_damage > route_claim
C002  material_damage  Moderate $  1,250      check_coverage > price_damage > route_claim
C003  total_loss       Major    $ 11,400      check_coverage > price_damage > route_claim
C004  total_loss       Major    $  5,450      check_coverage > price_damage > route_claim
C005  needs_review     not covered            check_coverage
C006  needs_review     policy not active      check_coverage


And the extraction gate, with typical bad model answers:

In [5]:
bad_outputs = [
    "Sure! The claim is about a windshield.",                                                      # chatty, not JSON
    '{"claim_id": "C002", "incident_type": "parking", "damage_areas": ["bumper"]}',                # unknown incident type
    '{"claim_id": "C002", "incident_type": "collision", "damage_areas": ["taillight"]}',           # not an allowed area
    '{"claim_id": "C004", "incident_type": "hail", "damage_areas": []}',                           # no areas
]
for raw in bad_outputs:
    try:
        ClaimFacts.model_validate_json(raw)
        print("✅ passed")
    except ValidationError as e:
        err = e.errors()[0]
        print(f"❌ {'.'.join(map(str, err['loc'])) or 'JSON'}: {err['msg'][:80]}")

❌ JSON: Invalid JSON: expected value at line 1 column 1
❌ incident_type: Input should be 'glass', 'collision', 'hail', 'vandalism' or 'theft'
❌ damage_areas.0: Input should be 'windshield', 'glass', 'front', 'rear', 'side', 'roof', 'hood', 
❌ damage_areas: List should have at least 1 item after validation, not 0


## 5 · Run the workflow

The same six claims as lesson P1. Compare `llm_calls` with the agent's `steps`.

In [6]:
sample_fnols = {
    "C001": "Claim ID: C001\nCustomer: John Smith\nVehicle: 2018 Toyota Camry\nIncident: While driving on the highway, "
            "a rock hit my windshield and caused a small chip about the size of a quarter. No other damage was observed.",
    "C002": "Claim ID: C002\nCustomer: Sarah Johnson\nVehicle: 2020 Honda Civic\nIncident: I was parked at the grocery store "
            "and returned to find someone had hit my car and dented the rear bumper and taillight. The taillight is broken "
            "and the bumper has a large dent.",
    "C003": "Claim ID: C003\nCustomer: Michael Rodriguez\nVehicle: 2022 Ford F-150\nIncident: I was involved in a serious "
            "collision at an intersection. The front of my truck is severely damaged, including the hood, bumper, radiator, "
            "and engine compartment. The airbags deployed and the vehicle is not drivable.",
    "C004": "Claim ID: C004\nCustomer: Emma Williams\nVehicle: 2019 Subaru Outback\nIncident: My car was damaged in a "
            "hailstorm. There are multiple dents on the hood, roof, and trunk. The side mirrors were also damaged and one "
            "window has a small crack.",
    "C005": "Claim ID: C005\nCustomer: David Brown\nVehicle: 2021 Tesla Model 3\nIncident: Someone keyed my car in the "
            "parking lot. There are deep scratches along both doors on the driver's side.",
    "C006": "Claim ID: C006\nCustomer: Priya Patel\nVehicle: 2017 Mazda 3\nIncident: I backed into a pole in a parking "
            "garage. The rear bumper is cracked and the trunk lid is dented.",
}

In [ ]:
for claim_id, text in sample_fnols.items():
    r = triage(text)
    detail = f"{r['severity']:8} ${r['est_cost']:>7,.0f}" if "severity" in r else r["reason"][:40]
    print(f"{claim_id}  {r['queue']:16} {detail:40} llm_calls={r['llm_calls']}")

## Your turn

1. Add a `theft` claim (car stolen). Where does the workflow break, and what branch does it need?
2. Add a step: claims that mention an injury go to a `bodily_injury` queue. Is that code, an LLM step, or both?
3. Count the branches after your changes. At what point would you rather have the agent from P1?